In [56]:
import sqlite3

conn = sqlite3.connect(r"/graph/jupyter/uit_law.db")
cursor = conn.cursor()

In [3]:
def get_parents_and_leaves(cursor):
    # Step 1: Fetch all laws
    cursor.execute("SELECT * FROM laws")
    rows = cursor.fetchall()
    columns = [col[0] for col in cursor.description]

    # Convert rows to dicts for easy access
    laws = [dict(zip(columns, row)) for row in rows]

    # Step 2: Identify parent IDs and leaf IDs
    all_ids = set(law["id"] for law in laws)
    parent_ids = set(law["parent_id"] for law in laws if law["parent_id"] is not None)
    leaf_ids = all_ids - parent_ids

    # Step 3: Collect parent rows
    parent_rows = [law for law in laws if law["id"] in parent_ids]

    # Step 4: Collect leaf rows, optionally including their parent chain
    leaf_rows_with_parents = []
    for leaf_id in leaf_ids:
        cursor.execute("SELECT * FROM laws WHERE id = ?", (leaf_id,))
        leaf = cursor.fetchone()
        if not leaf:
            continue
        leaf_dict = dict(zip(columns, leaf))
        chain = [leaf_dict]

        # Add parent chain
        current = leaf_dict
        while current['parent_id']:
            cursor.execute("SELECT * FROM laws WHERE id = ?", (current['parent_id'],))
            parent = cursor.fetchone()
            if not parent:
                break
            parent_dict = dict(zip(columns, parent))
            chain.insert(0, parent_dict)  # insert parent at the front
            current = parent_dict

        leaf_rows_with_parents.extend(chain)

    # Step 5: Combine parents and leaves, removing duplicates
    combined = {law['id']: law for law in parent_rows + leaf_rows_with_parents}
    return list(combined.values())

In [4]:
from graph.src.triplet_extraction import clean_text

def build_text_for_laws(law_rows, include_parent_content=True, skip_chapter_titles=True):
    # Find all leaf ids
    all_ids = set(law['id'] for law in law_rows)
    parent_ids = set(law['parent_id'] for law in law_rows if law['parent_id'] is not None)
    leaf_ids = all_ids - parent_ids

    results = []

    # Map for quick lookup
    id_to_law = {law['id']: law for law in law_rows}

    for leaf_id in leaf_ids:
        # Build parent → leaf chain
        chain = []
        current_id = leaf_id
        while current_id:
            law = id_to_law.get(current_id)
            if not law:
                break
            chain.insert(0, law)  # parent first
            current_id = law['parent_id']

        # Combine titles and content safely
        combined_title = ""
        combined_content = ""
        for law in chain:
            combined_title += (law.get('title') or "") + " "
            if include_parent_content or law['id'] == leaf_id:
                if not (skip_chapter_titles and ((law.get("title") or "").strip().lower().startswith("chương")
                        or (law.get("title") or "").strip().lower().startswith("điều"))):
                    combined_content += (law.get('content') or "") + "\n"

        leaf = chain[-1]
        results.append({
            "id": leaf['id'],
            "so_hieu": leaf.get('so_hieu'),
            "title": combined_title.strip(),
            "content": clean_text(combined_content.strip())
        })

    return results

In [5]:
rows = get_parents_and_leaves(cursor)
laws = build_text_for_laws(rows)

In [6]:
print(laws[1])

{'id': '4c1b284ff188794266e567f146e00cba1cc2384310a0416b6e6a419081087f73', 'so_hieu': '790/QĐ-ĐHCNTT', 'title': 'chương 1 điều 8 khoản 3', 'content': 'chương trình tiên tiến: được thiết kế, xây dựng dựa trên cơ sở chương trình đang được áp dụng ở trường đại học tiên tiến trên thế giới (gọi tắt là chương trình gốc) được giảng dạy bằng ngôn ngữ của chương trình gốc.'}


In [7]:
system = """
Transform a complex Vietnamese law sentence into a simple, independent sentence that contains at least one noun phrase (cụm danh từ) and one verb phrase (cụm động từ).

- First, analyze the given complex Vietnamese sentence and break it down to its core meaning by identifying which information is essential.
- Remove subordinate, or unnecessary clauses, as well as any parts that make the sentence dependent on another (ensure the resulting sentence can stand alone and fully conveys a clear idea).
- Ensure that the resulting simple sentence is grammatically independent (not subordinate or reliant on a previous clause for meaning) and contains:
    - At least one clear noun phrase (cụm danh từ)
    - At least one clear verb phrase (cụm động từ)
- Explicitly, reason step-by-step about which parts of the original complex sentence are essential before rewriting. Present your reasoning BEFORE you give the final simplified sentence.
- “Independent sentence” means the simplified sentence can be understood alone, without context from another clause or sentence.

# Steps
1. **Read the complex Vietnamese sentence provided.**
2. **Analyze the sentence carefully, breaking it into its essential components.**
3. **Step-by-step, decide which information must be included for meaning and which can be omitted to achieve simplicity. Justify each decision.**
4. **Assemble a new Vietnamese sentence that:**
    - Is grammatically independent (can stand alone).
    - Contains at least one noun phrase and one verb phrase.
5. **Present your reasoning BEFORE the final simplified sentence.**
6. **Output only the final simplified Vietnamese sentence seperate by coma and contain inside.**

# Output Format
- First, a clear step-by-step reasoning (in Vietnamese or English, as needed).
- Then, only the final simplified, independent Vietnamese sentence(s) as plain text.
- Do not include any additional explanation, code blocks, or formatting—just the reasoning and the sentence(s).

# Examples
**Example 1:**
Input: "người điều khiển phương tiện tham gia giao thông đường bộ không được dừng xe, đỗ xe tại các vị trí sau đây: che khuất biển báo hiệu đường bộ, đèn tín hiệu giao thông"
Reasoning: Câu gốc có hai hành động (“dừng xe” và “đỗ xe”) kết hợp bởi dấu phẩy. Để đơn giản hóa, tách từng hành động thành câu riêng biệt, mỗi câu đều có đủ chủ ngữ (“người điều khiển phương tiện tham gia giao thông đường bộ”) và vị ngữ, đồng thời nội dung có thể hiểu độc lập.
Output:
[
Người điều khiển phương tiện tham gia giao thông đường bộ không được dừng xe tại các vị trí che khuất biển báo hiệu đường bộ, đèn tín hiệu giao thông.
Người điều khiển phương tiện tham gia giao thông đường bộ không được đỗ xe tại các vị trí che khuất biển báo hiệu đường bộ, đèn tín hiệu giao thông.
]

**Example 2:**
Input: "Đối với các công trình đang sử dụng nhưng chưa có quy trình bảo trì, chủ sở hữu hoặc người quản lý, sử dụng công trình có trách nhiệm tổ chức lập quy trình bảo trì công trình đường bộ."
Reasoning: Mệnh đề phụ “đối với…” là điều kiện. Để câu đơn giản hóa và độc lập, giữ lại ý chính về trách nhiệm và đối tượng áp dụng, bỏ các mệnh đề phụ thừa.
Output:
[
Chủ sở hữu hoặc người quản lý, sử dụng công trình phải tổ chức lập quy trình bảo trì công trình đường bộ.
]

- Keep as much context as possible.
- If the original sentence contains lists or multiple actions, split them as needed to keep resulting sentences simple and independent.
- Fill in missing reasoning or outputs when examples are incomplete.
- Return the final simplified, independent Vietnamese sentence(s) as plain text inside square brackets.

(Reminder: The objective is to transform a complex Vietnamese sentence into an independent, simple sentence that contains at least one noun phrase and one verb phrase, using careful step-by-step reasoning BEFORE providing the final output. Output only the simplified Vietnamese sentence.)
"""

In [10]:
import json
from graph.src.triplet_extraction import init_gpt

gpt_client = init_gpt()
tasks = []
for law in laws:
    tasks.append({
        "custom_id": law["id"],
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": "gpt-4.1-mini",
            "temperature": 0.1,
            "messages": [
                {"role": "system", "content": system},
                {"role": "user", "content": f"Sentence that need simplified: {law['content']}"},
            ]
        }
    })

with open("batch_input_v3.jsonl", "w", encoding="utf8") as f:
    for t in tasks:
        f.write(json.dumps(t) + "\n")

In [12]:
import random
import json

# pick a random task
sample = random.choice(tasks)

# print in readable (pretty) JSON format
print(json.dumps(sample, indent=4, ensure_ascii=False))

{
    "custom_id": "3c870ad5b03f7b556777122ce15e01f592b5840e82b5b8a649f43a8b9bf8a9af",
    "method": "POST",
    "url": "/v1/chat/completions",
    "body": {
        "model": "gpt-4.1-mini",
        "temperature": 0.1,
        "messages": [
            {
                "role": "system",
                "content": "\nTransform a complex Vietnamese law sentence into a simple, independent sentence that contains at least one noun phrase (cụm danh từ) and one verb phrase (cụm động từ).\n\n- First, analyze the given complex Vietnamese sentence and break it down to its core meaning by identifying which information is essential.\n- Remove subordinate, or unnecessary clauses, as well as any parts that make the sentence dependent on another (ensure the resulting sentence can stand alone and fully conveys a clear idea).\n- Ensure that the resulting simple sentence is grammatically independent (not subordinate or reliant on a previous clause for meaning) and contains:\n    - At least one clear no

In [13]:
batch_file = gpt_client.files.create(
    file=open("batch_input_v3.jsonl", "rb"),
    purpose="batch"
)
file_id = batch_file.id
print("Uploaded file id:", file_id)

batch_job = gpt_client.batches.create(
    input_file_id=file_id,
    endpoint="/v1/chat/completions",
    completion_window="24h"
)
job_id = batch_job.id
print("Batch job id:", job_id)

Uploaded file id: file-42F9DSmRMjAHqbEgie1q9r
Batch job id: batch_69116f47c8408190a868cbbb0a7c8e19


In [15]:
import time

while True:
    job = gpt_client.batches.retrieve(job_id)
    print("Status:", job.status)
    if job.status == "completed":
        print("Done!")
        break
    elif job.status in ("failed", "cancelled"):
        raise Exception("Batch job failed: " + str(job))
    time.sleep(10)  # wait 30 seconds before checking again

Status: completed
Done!


In [16]:
import json

output_file_id = job.output_file_id
# note: for some SDK versions or endpoints you might call .text instead of .content
file_resp = gpt_client.files.content(output_file_id)
raw = file_resp.content.decode("utf8")  # or .text if appropriate

results = []
for line in raw.strip().split("\n"):
    obj = json.loads(line)
    # you might want to extract more fields, e.g., the content of the message:
    content = obj["response"]["body"]["choices"][0]["message"]["content"]
    results.append({
        "custom_id": obj.get("custom_id"),
        "response": content
    })

In [21]:
from graph.src.triplet_extraction import clean_text
from graph.src.db import extract_from_sqlite
def get_the_source(cursor, id):
    cursor.execute("SELECT title, content FROM laws WHERE id = ?", (id,))
    rows = cursor.fetchall()

    title = ""
    sentence = ""
    for r in rows:
        t = r[0] or ""    # r[0] is title
        c = r[1] or ""    # r[1] is content

        title += t + " "
        if not t.strip().startswith("Chương") and not t.strip().startswith("Điều"):
            sentence += t + " " + c + "\n"

    return title, sentence

In [22]:
# Now print first 50 items as readable JSON
for item in results[:50]:
    print("-------------------------")
    print(item["custom_id"])
    print(get_the_source(cursor, item["custom_id"]))
    print("Response: ")
    parts = item["response"].split("\n")
    can_print = True

    for part in parts:
        if not part:
            can_print = True
        if can_print:
            print(part)

-------------------------
fcd0071c966d3c3481709ed772930b1d5f59ff45af0873f9de3af95fed92286c
('khoản 2 ', 'khoản 2 Quy chế này áp dụng đối với các đơn vị, cá nhân liên quan trong đào tạo hệ đại học chính quy.\n')
Response: 
Reasoning:
- The original sentence contains a main clause "quy chế này áp dụng đối với các đơn vị, cá nhân liên quan trong đào tạo hệ đại học chính quy."
- The phrase "đối với các đơn vị, cá nhân liên quan trong đào tạo hệ đại học chính quy" is a prepositional phrase specifying the scope of application.
- The sentence is already quite simple and independent.
- It contains a noun phrase "quy chế này" and a verb phrase "áp dụng đối với..."
- No subordinate clauses or unnecessary information to remove.
- Therefore, the sentence can be kept as is for simplicity and independence.

[Quy chế này áp dụng đối với các đơn vị, cá nhân liên quan trong đào tạo hệ đại học chính quy.]
-------------------------
4c1b284ff188794266e567f146e00cba1cc2384310a0416b6e6a419081087f73
('khoản 

In [54]:
from graph.src.triplet_extraction import clean_text
from graph.src.db import extract_from_sqlite

rows = extract_from_sqlite(cursor, "f9d85a62ee93f79b0622db2d8a1a60df0f8d48aedb6a0bca56bf6d1327e199a7", True)
sentence = ""
title = ""
for r in rows:
    title += r['title'] + " "
    if not (r['title'].strip().startswith("Chươngf")):
        sentence += r['title'] + ": " + r['content'] + "\n"

so_hieu = r['so_hieu']
ID = r['id']
print(ID)
print(so_hieu + " " + title)
print(clean_text(sentence))

f9d85a62ee93f79b0622db2d8a1a60df0f8d48aedb6a0bca56bf6d1327e199a7
790/QĐ-ĐHCNTT chương 3 điều 26 khoản 3 điểm a 
chương 3: kiểm tra và thi học phần điều 26: công nhận và chuyển đổi tín chỉ khoản 3: việc công nhận và chuyển đổi tín chỉ của trường đảm bảo các nguyên tắc sau: điểm a: kết quả đối sánh chuẩn đầu ra, chương trình đào tạo và nội dung môn học là cơ sở cốt lõi cho việc công nhận và chuyển đổi tín chỉ;


In [51]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS laws_process (
    id TEXT,
    part INTEGER,
    content TEXT,
    so_hieu TEXT,
    PRIMARY KEY (id, part)
)
""")
conn.commit()

In [57]:
cursor.execute("DELETE FROM laws_process")
conn.commit()

In [58]:
import re

def extract_bracket_content(text):
    # Returns a list of all content inside square brackets []
    return re.findall(r'\[(.*?)\]', text, re.DOTALL)
count = 0
# Iterate over results
for item in results:
    custom_id = item["custom_id"]
    response_text = item.get("response", "")

    # Extract all bracketed content
    bracket_contents = extract_bracket_content(response_text)
    # print(bracket_contents)

    if not bracket_contents:
        print(f"[WARNING] No bracketed content found for custom_id={custom_id}")
        print(response_text)
        print("--------------------------------------------------")
        continue

    sentences = []
    for content in bracket_contents:
        # Split by '.', ';', or '\n'
        parts = re.split(r'[.;\n]', content)
        # Clean up and remove empty strings
        parts_clean = [p.strip() for p in parts if p.strip()]
        sentences.extend(parts_clean)

    if not sentences:
        print(f"[WARNING] No content found for custom_id={custom_id}")
        print(response_text)
        print("--------------------------------------------------")
        continue

    # Insert each sentence as a separate part
    for idx, sentence in enumerate(sentences, start=1):
        sentence = sentence.strip().replace('"', '')
        if not sentence:
            continue
        cursor.execute("""
        INSERT OR REPLACE INTO laws_process (id, part, content, so_hieu)
        VALUES (?, ?, ?, ?)
        """, (custom_id, idx, sentence, so_hieu))
        count += 1
        conn.commit()
print(f"Processed {count} sentences.")

[WARNING] No bracketed content found for custom_id=1ec58bb1788c1353879f4453b5121106ff89982c29004cd52fdfe11c6593a58c
Please provide the complex Vietnamese sentence you want me to simplify.
--------------------------------------------------
[WARNING] No bracketed content found for custom_id=6e0ad8a73f7a0f4d9ca6564b466f055544738394e88ff7c0a825e465119622ca
Please provide the complex Vietnamese sentence you want me to simplify.
--------------------------------------------------
[WARNING] No bracketed content found for custom_id=9a767f81033a259684b7485894f639be1c76c1b4adbcef617479f71aba03ffca
Bạn vui lòng cung cấp câu tiếng Việt phức tạp cần được đơn giản hóa để tôi có thể giúp bạn phân tích và viết lại câu đó.
--------------------------------------------------
[WARNING] No bracketed content found for custom_id=9dca1aff1f767a6dbeb54f6b6fe0bd21be639ceee62df7770a180832fb0fff12
Please provide the complex Vietnamese sentence you want me to simplify.
----------------------------------------------